# 02 - Model comparison

Train the four single-input architectures (MLP, CNN, ResNet, and Transformer) on the Amikacin task, compare their test accuracy and AUROC, and average three of them in an ensemble.

Each classifier has a `from_spectrum(bin_width, input_dim, **overrides)` factory that scales the architectural defaults (kernel sizes, for instance) to a given binning.

Uses the **MALDI-Kleb-AI** dataset (Rocchi *et al.*, 2026; [Zenodo DOI 10.5281/zenodo.17405072](https://zenodo.org/records/17405072)); see notebook 01 for caching details.

## 1. Load the demo dataset

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make notebooks/_demo.py importable
from notebooks._demo import binary_labels, load_maldi_kleb_ai

demo = load_maldi_kleb_ai(antibiotic='Amikacin')
X, y = binary_labels(demo)  # y == 1 for resistant
print(f'all spectra: {demo.X.shape} | labeled: {X.shape} '
      f'| prevalence(R): {y.mean():.2%}')
print(demo.batch.value_counts().to_string())

all spectra: (741, 6000) | labeled: (741, 6000) | prevalence(R): 49.80%
Batch
Rome       470
Milan      184
Catania     87


## 2. Fit the four architectures

Each architecture is built by `from_spectrum` for the 3 Da bins of this dataset and trained for 30 epochs on a stratified 75% split.

**GPU recommended.** This notebook takes a few minutes on a GPU and about half an hour on a CPU, and the outputs below come from a GPU. On a GPU the convolutions of the CNN and the ResNet are not exactly repeatable, so `seed_everything(0, deterministic=True)` switches PyTorch to its deterministic algorithms, and the memory-efficient attention kernel, which has no deterministic version, is turned off for the Transformer: a new run on the same GPU gives the same numbers.

In [2]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0,
)
input_dim = X.shape[1]

In [3]:
import torch
from sklearn.metrics import roc_auc_score

from maldideepkit import (
    MaldiCNNClassifier,
    MaldiMLPClassifier,
    MaldiResNetClassifier,
    MaldiTransformerClassifier,
)
from maldideepkit.utils import seed_everything

seed_everything(0, deterministic=True)
torch.backends.cuda.enable_mem_efficient_sdp(False)  # deterministic attention

common = dict(bin_width=3, input_dim=input_dim, epochs=30, random_state=0)
classifiers = {
    'MLP': MaldiMLPClassifier.from_spectrum(**common),
    'CNN': MaldiCNNClassifier.from_spectrum(**common),
    'ResNet': MaldiResNetClassifier.from_spectrum(**common),
    'Transformer': MaldiTransformerClassifier.from_spectrum(**common),
}

for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)
    auroc = roc_auc_score(y_te, proba[:, 1])
    print(f'{name:<12s} accuracy: {clf.score(X_te, y_te):.3f} | AUROC: {auroc:.3f}')

MLP          accuracy: 0.747 | AUROC: 0.826


CNN          accuracy: 0.500 | AUROC: 0.587


ResNet       accuracy: 0.677 | AUROC: 0.718


Transformer  accuracy: 0.500 | AUROC: 0.498


**Small dataset, short schedule.** With about 555 training spectra and 30 epochs, the ResNet and the Transformer stay near chance, and the CNN result changes from one device to another. Read this comparison as a demonstration of the shared interface, not as a ranking of the architectures.

## 3. Mean-of-probabilities ensemble

`SpectralEnsemble` fits each member on the same data and averages their `predict_proba` outputs. Here the members are an MLP, a CNN, and a ResNet, each with its own seed.

In [4]:
from maldideepkit.utils import SpectralEnsemble

ensemble = SpectralEnsemble([
    MaldiMLPClassifier(input_dim=input_dim, epochs=30, random_state=0),
    MaldiCNNClassifier(input_dim=input_dim, epochs=30, random_state=1),
    MaldiResNetClassifier(input_dim=input_dim, epochs=30, random_state=2),
]).fit(X_tr, y_tr)

proba = ensemble.predict_proba(X_te)
accuracy = (proba.argmax(axis=1) == y_te).mean()
auroc = roc_auc_score(y_te, proba[:, 1])
print(f'{"Ensemble":<12s} accuracy: {accuracy:.3f} | AUROC: {auroc:.3f}')

Ensemble     accuracy: 0.656 | AUROC: 0.786


## Recap

The four architectures share one interface, and `from_spectrum` sizes each of them for a given binning. `SpectralEnsemble` averages the probabilities of its members. See the [API reference](https://maldideepkit.readthedocs.io/en/latest/api/index.html) for each classifier and the [utilities page](https://maldideepkit.readthedocs.io/en/latest/api/utils.html) for `SpectralEnsemble`.